# 03 — CBPE: Performance Estimation Without Labels

Confidence-Based Performance Estimation (CBPE) estimates classification metrics on unlabelled windows by treating calibrated probabilities as fractional certainties.

This notebook demonstrates CBPE using a **known generating process** so we can compare all five estimated metrics against their true supervised counterparts.

Four scenarios cover the main production situations:

| Scenario | What changes | PSI fires? | CBPE tracks? |
|---|---|---|---|
| **A — Covariate shift** | P(X) shifts; boundary stable | ✓ yes | ✓ yes |
| **B — Boundary inversion** | labels inverted; P(X) fixed | ✗ no | ✗ blind |
| **C — Boundary shift** | threshold 0 → +1.5; P(X) fixed | ✗ no | ✗ blind |
| **D — Feature weight shift** | X2 weight 1 → 5; P(X) fixed | ✗ no | ✗ blind |

Section 3 proves that CBPE's blindness to B–D is structural, not coincidental.

In [1]:
%load_ext autoreload
%autoreload 2

## 1. Generating process

The ground truth is a linear boundary:

```
Y = 1   if   X1 + x2_weight * X2 > threshold   (with Bernoulli noise)
```

Default: `x2_weight=1.0`, `threshold=0.0` — a symmetric, balanced task.
Both parameters are varied per-scenario to simulate concept drift.

In [2]:
import logging

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

logging.getLogger("ayn_ml").setLevel(logging.ERROR)

N_REF = 2_000
N_CUR = 500


def make_data(n, x1_mean=0.0, invert_labels=False, threshold=0.0, x2_weight=1.0, seed=None):
    """Generate binary classification data with a configurable linear boundary.

    Decision boundary: X1 + x2_weight * X2 > threshold.
    """
    rng = np.random.default_rng(seed)
    X1 = rng.normal(x1_mean, 1.0, size=n)
    X2 = rng.normal(0.0, 1.0, size=n)
    score = X1 + x2_weight * X2 - threshold
    if invert_labels:
        score = -score
    prob_true = 1 / (1 + np.exp(-score))
    Y = rng.binomial(1, prob_true)
    return pd.DataFrame({"X1": X1, "X2": X2, "y_true": Y})


df_ref = make_data(N_REF, seed=0)
print("Reference — shape:", df_ref.shape, "  positive rate:", df_ref.y_true.mean().round(3))

Reference — shape: (2000, 3)   positive rate: 0.514


## 2. Train a fixed model on the reference window

This model is trained once and never retrained. In production this mirrors a deployed model that stays fixed while the world changes around it.

In [3]:
model = LogisticRegression(random_state=42)
model.fit(df_ref[["X1", "X2"]], df_ref["y_true"])

df_ref["y_pred"] = model.predict(df_ref[["X1", "X2"]])
df_ref["y_pred_proba"] = model.predict_proba(df_ref[["X1", "X2"]])[:, 1]

ref_acc = accuracy_score(df_ref["y_true"], df_ref["y_pred"])
print(f"Reference accuracy (true, in-sample):  {ref_acc:.3f}")

Reference accuracy (true, in-sample):  0.714


## 3. Why CBPE is blind to concept drift — mathematical proof

On the **current window**, CBPE uses only model outputs:

```
p̂ᵢ = f(Xᵢ)   — calibrated probability (function of X only)
ŷᵢ = g(Xᵢ)   — hard prediction        (function of X only)
```

The accuracy estimate expands to:

```
CBPE_accuracy = (1/N) Σᵢ [ p̂ᵢ · I(ŷᵢ=1)  +  (1 − p̂ᵢ) · I(ŷᵢ=0) ]
```

**Y never appears.** The reference window provides `y_true` only to fit the isotonic calibrator — a one-time step that maps raw probabilities to calibrated ones. Once fitted, the calibrator is a deterministic function of X.

**Formal consequence:** fix P(X). Then the sample `{Xᵢ}` follows the same distribution across scenarios B, C, and D. The model outputs `{p̂ᵢ, ŷᵢ}` are deterministic functions of `{Xᵢ}`, so their distribution is unchanged. Therefore:

> CBPE is **provably invariant to any change in P(Y|X) when P(X) is fixed**.

This covers label inversions (B), boundary shifts (C), and feature re-weighting (D) equally.

The other four metrics follow the same logic: F1, precision, and recall derive from a fractional confusion matrix built from `{p̂ᵢ, ŷᵢ}` only; AUC sweeps thresholds over `{p̂ᵢ}` only. None of them reads `Yᵢ` from the current window. **The invariance proof applies to all five CBPE metrics equally.**

In [4]:
from ayn_ml.core.schema import TabularSchema
from ayn_ml.core.spec import MetricSpec
from ayn_ml.metrics import get_metric

SCHEMA = TabularSchema(
    label_col="y_true",
    prediction_col="y_pred",
    proba_col="y_pred_proba",
)

_PAIRS = [
    ("accuracy", "cbpe_accuracy"),
    ("auc", "cbpe_auc"),
    ("f1", "cbpe_f1"),
    ("precision", "cbpe_precision"),
    ("recall", "cbpe_recall"),
]


def evaluate(df_current, df_reference, label):
    """Compute supervised and CBPE estimates for all five metrics, plus PSI."""
    record = {"scenario": label}
    rows = []
    for true_name, cbpe_name in _PAIRS:
        true_val = get_metric(true_name).compute(df_current, None, SCHEMA, MetricSpec(name=true_name)).value
        cbpe_val = get_metric(cbpe_name).compute(df_current, df_reference, SCHEMA, MetricSpec(name=cbpe_name)).value
        rows.append(
            {
                "metric": true_name,
                "supervised": round(true_val, 3),
                "CBPE est.": round(cbpe_val, 3),
                "Δ (CBPE−true)": round(cbpe_val - true_val, 3),
            }
        )
        record[true_name] = {
            "supervised": round(true_val, 3),
            "cbpe": round(cbpe_val, 3),
            "delta": round(cbpe_val - true_val, 3),
        }

    psi_x1 = (
        get_metric("psi").compute(df_current, df_reference, SCHEMA, MetricSpec(name="psi", feature_name="X1")).value
    )
    psi_x2 = (
        get_metric("psi").compute(df_current, df_reference, SCHEMA, MetricSpec(name="psi", feature_name="X2")).value
    )
    record["psi_x1"] = round(psi_x1, 3)
    record["psi_x2"] = round(psi_x2, 3)

    df_out = pd.DataFrame(rows).set_index("metric")
    sep = "─" * max(1, 56 - len(label))
    print(f"── {label} {sep}")
    print(df_out.to_string())
    print(f"   PSI X1={psi_x1:.3f}   PSI X2={psi_x2:.3f}\n")

    return record

## 4. Baseline — same distribution as reference

Sanity check: with no shift, CBPE estimates should be close to the true supervised metrics.

In [5]:
df_baseline = make_data(N_CUR, seed=10)
df_baseline["y_pred"] = model.predict(df_baseline[["X1", "X2"]])
df_baseline["y_pred_proba"] = model.predict_proba(df_baseline[["X1", "X2"]])[:, 1]

row_baseline = evaluate(df_baseline, df_ref, "Baseline (no shift)")

── Baseline (no shift) ─────────────────────────────────────
           supervised  CBPE est.  Δ (CBPE−true)
metric                                         
accuracy        0.694      0.715          0.021
auc             0.763      0.797          0.034
f1              0.694      0.703          0.009
precision       0.694      0.725          0.031
recall          0.694      0.682         -0.012
   PSI X1=0.013   PSI X2=0.029



## 5. Scenario A — Covariate shift

`X1` mean shifts from 0 to +1.5. The decision boundary `X1 + X2 > 0` is **unchanged** — the same rule governs labels. The model's probability assignments remain well-calibrated relative to the new distribution.

Expected: CBPE tracks the true metrics because P(Y|X) has not changed.

In [6]:
df_covariate = make_data(N_CUR, x1_mean=1.5, seed=20)
df_covariate["y_pred"] = model.predict(df_covariate[["X1", "X2"]])
df_covariate["y_pred_proba"] = model.predict_proba(df_covariate[["X1", "X2"]])[:, 1]

print("X1 mean in current window:", df_covariate.X1.mean().round(2), "  (reference: ~0.0)\n")

row_covariate = evaluate(df_covariate, df_ref, "Scenario A — Covariate shift (X1 mean +1.5)")

X1 mean in current window: 1.41   (reference: ~0.0)

── Scenario A — Covariate shift (X1 mean +1.5) ─────────────
           supervised  CBPE est.  Δ (CBPE−true)
metric                                         
accuracy        0.756      0.797          0.041
auc             0.781      0.844          0.063
f1              0.734      0.871          0.137
precision       0.737      0.828          0.091
recall          0.756      0.920          0.164
   PSI X1=1.918   PSI X2=0.032



**What to observe:**
- PSI X1 is elevated — distribution has drifted.
- True metrics and CBPE estimates remain close — CBPE correctly tracks performance because P(Y|X) is unchanged.
- This is CBPE's intended use case: covariate shift is detectable by PSI, and CBPE provides a reliable performance estimate without labels.

## 6. Scenario B — Concept drift: boundary inversion

Feature distribution is **identical** to reference. Only the decision boundary flips:

```
Before:   Y = 1  if  X1 + X2 > 0      (trained model)
After:    Y = 1  if  -(X1 + X2) > 0   (inverted boundary)
```

The model still outputs the same probabilities. But those probabilities are now **wrong** — observations the model is confident about are now the opposite class.

Expected:
- True accuracy collapses toward `1 − ref_accuracy` (model is almost always wrong).
- CBPE stays near `ref_accuracy` (it cannot see the label flip).
- PSI X1 and PSI X2 ≈ 0 (inputs unchanged).

In [7]:
df_concept = make_data(N_CUR, invert_labels=True, seed=30)
df_concept["y_pred"] = model.predict(df_concept[["X1", "X2"]])
df_concept["y_pred_proba"] = model.predict_proba(df_concept[["X1", "X2"]])[:, 1]

print("X1 mean:", df_concept.X1.mean().round(2), "  (should be ~0.0)\n")

row_concept = evaluate(df_concept, df_ref, "Scenario B — Boundary inversion")

X1 mean: 0.04   (should be ~0.0)

── Scenario B — Boundary inversion ─────────────────────────
           supervised  CBPE est.  Δ (CBPE−true)
metric                                         
accuracy        0.278      0.717          0.439
auc             0.215      0.799          0.584
f1              0.276      0.737          0.461
precision       0.276      0.718          0.442
recall          0.278      0.757          0.479
   PSI X1=0.015   PSI X2=0.006



**What to observe:**
- PSI X1 and PSI X2 ≈ 0 — no covariate shift.
- True accuracy ≈ `1 − ref_accuracy` — the model is confidently wrong on almost every prediction.
- CBPE accuracy stays near `ref_accuracy` — completely blind.
- For AUC: the supervised AUC collapses toward 0 (rank ordering inverted); CBPE AUC stays near the reference value — the largest possible gap.

## 7. Scenario C — Concept drift: boundary shift

The threshold moves from 0 to **+1.5**:

```
Before:   Y = 1  if  X1 + X2 > 0       (trained model)
After:    Y = 1  if  X1 + X2 > 1.5
```

Fewer observations now qualify as positive (the class becomes rarer). The model still scores observations relative to the original threshold, so it over-predicts class 1 in the 0 < X1+X2 < 1.5 band.

Expected: true accuracy drops, CBPE stays near baseline, PSI ≈ 0.

In [8]:
df_partial = make_data(N_CUR, threshold=1.5, seed=40)
df_partial["y_pred"] = model.predict(df_partial[["X1", "X2"]])
df_partial["y_pred_proba"] = model.predict_proba(df_partial[["X1", "X2"]])[:, 1]

print("X1 mean:", df_partial.X1.mean().round(2), "  (reference: ~0.0)")
print("Positive rate:", df_partial.y_true.mean().round(3), "  (reference: ~0.5)")
print("(fewer positives because boundary shifted right)\n")

row_partial = evaluate(df_partial, df_ref, "Scenario C — Boundary shift (threshold +1.5)")

X1 mean: -0.01   (reference: ~0.0)
Positive rate: 0.24   (reference: ~0.5)
(fewer positives because boundary shifted right)

── Scenario C — Boundary shift (threshold +1.5) ────────────
           supervised  CBPE est.  Δ (CBPE−true)
metric                                         
accuracy        0.648      0.727          0.079
auc             0.806      0.812          0.006
f1              0.673      0.738          0.066
precision       0.803      0.730         -0.073
recall          0.648      0.747          0.099
   PSI X1=0.016   PSI X2=0.015



**What to observe:**
- PSI X1 and PSI X2 ≈ 0 — same feature distribution.
- Positive rate drops from ~0.5 — the boundary moved right.
- True accuracy drops; CBPE stays near the baseline — same model outputs, same estimate.
- The drift is not a dramatic inversion, just a gradual shift — CBPE is equally blind.

## 8. Scenario D — Concept drift: feature weight shift

The feature weights change: the true boundary shifts from `X1 + X2 > 0` to `X1 + 5·X2 > 0`.
X2 becomes five times more predictive than X1, but the model still scores observations as `X1 + X2`.

This is a realistic concept drift — a previously minor signal becomes dominant.
P(X) is unchanged, so the math predicts CBPE remains flat.

In [9]:
df_weight = make_data(N_CUR, x2_weight=5.0, seed=50)
df_weight["y_pred"] = model.predict(df_weight[["X1", "X2"]])
df_weight["y_pred_proba"] = model.predict_proba(df_weight[["X1", "X2"]])[:, 1]

print("X1 mean:", df_weight.X1.mean().round(2), "  (reference: ~0.0)")
print("Positive rate:", df_weight.y_true.mean().round(3), "  (reference: ~0.5)\n")

row_weight = evaluate(df_weight, df_ref, "Scenario D — Feature weight shift (X1 + 5·X2)")

X1 mean: 0.02   (reference: ~0.0)
Positive rate: 0.498   (reference: ~0.5)

── Scenario D — Feature weight shift (X1 + 5·X2) ───────────
           supervised  CBPE est.  Δ (CBPE−true)
metric                                         
accuracy        0.762      0.713         -0.049
auc             0.852      0.796         -0.056
f1              0.762      0.726         -0.036
precision       0.763      0.722         -0.041
recall          0.762      0.731         -0.031
   PSI X1=0.066   PSI X2=0.031



**What to observe:**
- PSI X1 and PSI X2 ≈ 0 — same feature distribution.
- Positive rate ≈ 0.5 (X2 is symmetric, so the new boundary is still balanced).
- Model scores using equal weights — it is misaligned in quadrants where X2 dominates.
- CBPE stays near the baseline; the delta between supervised and CBPE reflects the model's miscalibration relative to the new truth.
- The type of drift (inversion vs shift vs reweighting) doesn't matter: CBPE is blind to all of them when P(X) is fixed.

## 9. Summary — delta table

Columns show `Δ = CBPE estimate − true supervised value` for each metric.
The colormap is centred at 0 (yellow):
- **Green (negative Δ):** CBPE underestimates — supervised is better than CBPE thinks.
- **Yellow (near 0):** CBPE tracks reality well.
- **Red (positive Δ):** CBPE overestimates — apparent performance is illusory.

In [10]:
metrics = ["accuracy", "auc", "f1", "precision", "recall"]
_results = [row_baseline, row_covariate, row_concept, row_partial, row_weight]

delta_rows = []
for r in _results:
    row = {"scenario": r["scenario"], "PSI X1": r["psi_x1"], "PSI X2": r["psi_x2"]}
    for m in metrics:
        row[f"Δ {m}"] = r[m]["delta"]
    delta_rows.append(row)

df_delta = pd.DataFrame(delta_rows).set_index("scenario")
delta_cols = [f"Δ {m}" for m in metrics]
df_delta.style.background_gradient(
    cmap="RdYlGn_r",
    subset=delta_cols,
    vmin=-0.3,
    vmax=0.3,
).format({c: "{:+.3f}" for c in delta_cols})

,PSI X1,PSI X2,Δ accuracy,Δ auc,Δ f1,Δ precision,Δ recall
scenario,,,,,,,
Baseline (no shift),0.013000,0.029000,+0.021,+0.034,+0.009,+0.031,-0.012
Scenario A — Covariate shift (X1 mean +1.5),1.918000,0.032000,+0.041,+0.063,+0.137,+0.091,+0.164
Scenario B — Boundary inversion,0.015000,0.006000,+0.439,+0.584,+0.461,+0.442,+0.479
Scenario C — Boundary shift (threshold +1.5),0.016000,0.015000,+0.079,+0.006,+0.066,-0.073,+0.099
Scenario D — Feature weight shift (X1 + 5·X2),0.066000,0.031000,-0.049,-0.056,-0.036,-0.041,-0.031


## 10. Takeaways

### The mathematical result

CBPE on the current window is a function of model outputs `{p̂ᵢ, ŷᵢ}` alone, which are deterministic functions of `{Xᵢ}`. It never reads `y_true` from the current window. Therefore, when P(X) is fixed, CBPE is constant — scenarios B, C, and D all produce the same structural blindness.

### When CBPE works
- **Covariate shift**: P(X) changes → model probabilities shift → CBPE tracks the accuracy change.
- **Data quality degradation**: missing features or added noise push probabilities toward 0.5 → CBPE captures the accuracy drop.

### When CBPE fails
- **Any concept drift with fixed P(X)**: label inversions, boundary shifts, feature reweighting — all invisible. This is not a corner case; it is provable.
- **Miscalibration**: if `calibrate=False` and the model is poorly calibrated, estimates are unreliable even without drift.

### Recommended monitoring stack

```
CBPE metrics          →  performance estimate when labels are delayed (covariate shift only)
PSI / Wasserstein     →  detect P(X) shift on key features
Target drift          →  PSI on y_true; fires when P(Y) shifts (e.g. Scenario C: rate 0.5→0.25)
                          silent when drift is symmetric (B: inversion keeps rate ≈0.5, D: balanced boundary)
Prediction drift      →  PSI on y_pred_proba; silent when P(X) is fixed (model outputs unchanged)
                          useful only in combination with rising true error rate
Labelled sample       →  ground truth; the only reliable defence against all concept drift
```

**Why target drift misses B and D:** P(Y) = ∫ P(Y|X)·P(X) dX. When the drift is symmetric around the class boundary (inverted boundary, symmetric weight shift), the integral stays at ~0.5 even though P(Y|X) has changed everywhere. Target drift is necessary but not sufficient for concept drift detection.